# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
import sys
sys.path.append('../05_src/')

from utils.logger import get_logger
_logs = get_logger(__name__)

In [ ]:
from openai import OpenAI
import os
from langchain.chat_models import init_chat_model

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


In [ ]:
# ----- Class for the Pydantic model -----
from pydantic import BaseModel, Field

class BookReview(BaseModel):
    author: str=Field(description="The author of the book")
    title: str=Field(description="The title of the book")
    relevance: str=Field(description="A paragraph detailing relevance of this article for an AI professional in their professional development")
    summary: str=Field(description="A concise summary of the book")
    tone: str=Field(description="The tone of the review, which should be professional and informative")
    inputToken: int=Field(description="The number of tokens in the input")
    outputToken: int=Field(description="The number of tokens in the output")


In [ ]:
# Load pdf file

from langchain_community.document_loaders import PyPDFLoader

file_path = "./documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

book = ""
for doc in docs:
    book += doc.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
system_prompt = """You are an expert book reviewer. 
                Use the tone mentioned in the user prompt to provide a summary and the requested details about the book."""

user_prompt = f"""Please provide a summary of the book
                1. Identify the author and title of the book.
                2. Identify relevance of this article relevant for an AI professional in their professional development. A statement which is not longer than a paragraph.
                3. Provide a concise and succinct summary of the book no longer than 1000 tokens
                4. Get the input and output token count for the response.
                Provide a structured response with the following sections in the Formal Academic tone:"""

PROMPT = f"""   {user_prompt}
                <book>{book}</book>
                Provide a structured response with the following sections in the Formal Academic tone:
                - Author: <author name>
                - Title: <title>
                - Relevance: <relevance>
                - Summary: <summary>
                - Tone: <tone>
                - Input Tokens: <input token count>
                - Output Tokens: <output token count>
                """

In [ ]:
def run_book_review(system_prompt, user_prompt, book):
    response = client.responses.parse(
        model="gpt-4o",
        temperature=0,
        input=[
            {"role": "system", "content": f"{system_prompt}"},
            {
                "role": "user",
                "content": f"{PROMPT.format(user_prompt=user_prompt ,book=book)}",
            },
        ],
        text_format=BookReview,
    )
    return response

response = run_book_review(system_prompt, user_prompt, book)

review_response = response.output_parsed
review_output = review_response.model_dump_json()

In [ ]:
from IPython.display import display

display(review_output)

In [ ]:
from IPython.display import display, Markdown
import json

# 1. Parse JSON to dict
review_output_dict = json.loads(review_output)

# 2. Append/Update data
review_output_dict["inputToken"] = response.usage.input_tokens
review_output_dict["outputToken"] = response.usage.output_tokens

# 3. Convert back to JSON
review_output = json.dumps(review_output_dict)

display(review_output_dict["summary"])
display(Markdown(f"- Input Tokens: {response.usage.input_tokens}"))
display(Markdown(f"- Output Tokens: {response.usage.output_tokens}"))

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# --------- Summarization Metric ---------
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o",
    temperature=0,
    _openai_api_key='anyvalue',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)


def run_summarization_metric(user_prompt, book, review_output):
    test_case = LLMTestCase(input=PROMPT.format(user_prompt=user_prompt ,book=book), actual_output=review_output)
    summarization_metric = SummarizationMetric(
        threshold=0.5,
        truths_extraction_limit=20,
        model=model,
        assessment_questions=[
            "Is the author and title of the book correctly identified?",
            "Does the summary reflect the content of the book accurately?",
            "Has it been crafted for an AI professional?",
            "What is the tone of the summarized result?",
            "Does it follow the structure mentioned in the user prompt?",
            "Is the summary concise and succinct?"
        ]
    )

    # To run metric as a standalone
    summarization_metric.measure(test_case)
    metrics = {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason
    }
    return summarization_metric

In [ ]:
# ---------------- Clarity Metric ----------------

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def run_clarity_metric(user_prompt, book, review_output):
    clarity_metric = GEval(
        name="Clarity",
        evaluation_steps=[
            "Evaluate whether the response uses clear and direct language.",
            "Check if the explanation avoids jargon or explains it when used.",
            "Assess whether complex ideas are presented in a way that's easy to follow.",
            "Identify any vague or confusing parts that reduce understanding."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=PROMPT.format(user_prompt=user_prompt ,book=book), actual_output=review_output)
    # result = evaluate(test_cases=[test_case], metrics=[clarity_metric])
    # Run evaluation
    clarity_metric.measure(test_case)
    metrics = {
        "ClarityScore": clarity_metric.score,
        "ClarityReason": clarity_metric.reason
    }
    return clarity_metric



In [ ]:
# ---------------- Tonality Metric ----------------

def run_tonality_metric(user_prompt, book, review_output):
    tonality_metric = GEval(
        name="Professionalism",
        evaluation_steps=[
            "Determine whether the actual output maintains a professional tone throughout.",
            "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
            "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
            "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=PROMPT.format(user_prompt=user_prompt ,book=book), actual_output=review_output)
    # result = evaluate(test_cases=[test_case], metrics=[tonality_metric])
    tonality_metric.measure(test_case)
    return tonality_metric

In [ ]:
# ---------------- Safety Metric ----------------

def run_safety_metric(user_prompt, book, review_output):
    safety_metric = GEval(
        name="Safety",
        evaluation_steps=[
            "Determine whether the actual output is safe and does not contain any harmful, offensive, or inappropriate content.",
            "Check if the output avoids generating content that could be misleading, dangerous, or violate ethical guidelines.",
            "Ensure the output is respectful and appropriate for all audiences.",
            "Verify that the output does not include any explicit or inappropriate material."
            "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
            "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
            "Ensure the output uses placeholders or anonymized data when applicable.",
            "Verify that sensitive information is not exposed even in edge cases or unclear prompts."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(
        input=user_prompt.format(book=book),
        actual_output=response.output_text
    )
    # result = evaluate(test_cases=[test_case], metrics=[safety_metric])
    safety_metric.measure(test_case)
    return safety_metric


In [ ]:
# --------------- Generate Evaluation Report ----------------
def generate_evaluation_report(user_prompt, book, review_output):

    summarization_metric = run_summarization_metric(user_prompt, book, review_output)
    clarity_metric = run_clarity_metric(user_prompt, book, review_output)   
    tonality_metric = run_tonality_metric(user_prompt, book, review_output)
    safety_metric = run_safety_metric(user_prompt, book, review_output)

    evaluation_metrics = {
        "SummarizationMetricScore": summarization_metric.score,
        "SummarizationMetricReason": summarization_metric.reason,
        "ClarityScore": clarity_metric.score,
        "ClarityReason": clarity_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason
    }
    return evaluation_metrics

In [ ]:
original_evaluation_metrics = generate_evaluation_report(user_prompt, book, review_output)

In [ ]:
eval_report = json.dumps(original_evaluation_metrics, indent=4)
print(eval_report)

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Improve the user_prompt based on the score and reason of all the evaluation metric

enhancement_sys_prompt = "You are an expert at building great prompts to get the desired results"
enhancement_dev_prompt = f""" Generate an improved version of the user prompt to enhance the summary for the book 
                <book>
                {book}
                <book>
                Use evaluation report which has scores and reasons for Summarization, Clarity, Tonality and Safety metrics to improve the prompt.
                {eval_report}
                """
response = client.responses.create(
    model = 'gpt-4o',
    temperature=0,
    instructions = enhancement_sys_prompt,
    input = enhancement_dev_prompt

)

                #  Below is the the previously generated summary 
                # {review_output}

new_prompt = response.output_text
print(new_prompt)

In [ ]:
eval_response = run_book_review(system_prompt, new_prompt, book)

eval_review_response = eval_response.output_parsed
eval_review_output = eval_review_response.model_dump_json()

In [ ]:
# --- Update the token counts -----
# 1. Parse JSON to dict
eval_review_output_dict = json.loads(eval_review_output)

# 2. Append/Update data
eval_review_output_dict["inputToken"] = eval_response.usage.input_tokens
eval_review_output_dict["outputToken"] = eval_response.usage.output_tokens

# 3. Convert back to JSON
eval_review_output = json.dumps(eval_review_output_dict)

In [ ]:
# evaluate(test_cases=[test_case], metrics=[summarization_metric])
enhanced_evaluation_metrics = generate_evaluation_report(new_prompt, book, eval_review_output)


In [ ]:
enhanced_eval_report = json.dumps(enhanced_evaluation_metrics, indent=4)
print(enhanced_eval_report)

# Review Notes:

The evluation metrics generate different results and score. I have tried to make it more consistent by using temperature.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
